# ONIA · 5.9 — Clustering, topic modeling și anomalii în text

Studiază fiecare experiment independent, schimbă datele și compară rezultatele. Exemplele pentru ONIA #266 sunt sintetice; problema oficială este clasificare supervizată.

## 1. Corpus minimal și documente neetichetate

Înaintea unui algoritm definim ce reprezintă un document și verificăm dacă textele sunt suficiente pentru a forma grupuri coerente.

In [ ]:
documente = [
    "grafuri noduri muchii drumuri",
    "grafuri bfs componente conexe",
    "programare dinamica memoizare",
    "dinamica subprobleme tranzitii",
    "sir caractere palindrom",
    "siruri prefix sufix"
]
for i, text in enumerate(documente):
    print(f"D{i}: {text}")


## 2. Bag-of-Words cu frecvențe întregi

Matricea are câte un rând pentru document și câte o coloană pentru termen. Observăm că matricea este în mare parte nulă.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texte = ["graf nod nod", "graf drum", "dinamica stare"]
v = CountVectorizer()
X = v.fit_transform(texte)
print(v.get_feature_names_out())
print(X.toarray())


## 3. TF-IDF pentru un corpus de probleme algoritmice

TF-IDF reduce ponderea termenilor foarte răspândiți; bigramele păstrează unele expresii locale.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texte = [
    "graf nod drum BFS", "graf nod DFS", "dinamica stare memoizare",
    "dp subprobleme memoizare", "sir caractere prefix", "string sufix pattern"
]
v = TfidfVectorizer(ngram_range=(1, 2), norm="l2")
X = v.fit_transform(texte)
print("Forma:", X.shape)
print("Primele 12 caracteristici:", v.get_feature_names_out()[:12])


## 4. Calculul manual al distanței euclidiene

Nu avem nevoie de radical pentru a compara două distanțe euclidiene; ordinea rămâne aceeași.

In [ ]:
def distanta2(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b))

p = [1, 2]
centre = [[0, 0], [3, 2]]
for j, c in enumerate(centre):
    print(j, distanta2(p, c))
print("Cluster ales:", min(range(len(centre)), key=lambda j: distanta2(p, centre[j])))


## 5. O iterație K-Means scrisă de la zero

Se văd separat cele două operații: atribuirea la centrul apropiat și recalcularea mediei fiecărui grup.

In [ ]:
puncte = [[0, 0], [0, 2], [5, 4], [6, 5]]
centre = [[0, 1], [5, 5]]

def dist2(a, b):
    return sum((a[i] - b[i]) ** 2 for i in range(len(a)))

etichete = [min(range(len(centre)), key=lambda j: dist2(p, centre[j])) for p in puncte]
centre_noi = []
for j in range(len(centre)):
    grup = [p for p, e in zip(puncte, etichete) if e == j]
    centru = [sum(p[d] for p in grup) / len(grup) for d in range(2)]
    centre_noi.append(centru)
print("Atribuiri:", etichete)
print("Centre noi:", centre_noi)


## 6. KMeans pe TF-IDF și distribuția clusterelor

ID-urile 0, 1, 2 sunt arbitrare. Clusterele se interpretează din documentele și termenii lor, nu din indice.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

texte = [
    "graf bfs noduri", "graf dfs muchii", "graf drumuri", 
    "dp stare memoizare", "dinamica subprobleme", "dp tranzitii",
    "siruri palindrom", "sir caractere sufix", "cautare pattern"
]
X = TfidfVectorizer().fit_transform(texte)
m = KMeans(n_clusters=3, n_init=20, random_state=7)
etichete = m.fit_predict(X)
for i, e in enumerate(etichete):
    print(i, "cluster", e, texte[i])
print("Mărimi:", np.bincount(etichete))


## 7. Termenii cei mai importanți ai centrelor

Topul de termeni propune o interpretare, dar nu dovedește automat că tema este corectă. E necesară inspectarea documentelor.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

texte = ["tenis meci sport", "fotbal stadion sport", "python algoritm cod", "python date cod", "reteta prajitura", "reteta supa"]
v = TfidfVectorizer()
X = v.fit_transform(texte)
m = KMeans(n_clusters=3, n_init=20, random_state=2).fit(X)
termeni = v.get_feature_names_out()
for j, centru in enumerate(m.cluster_centers_):
    indici = centru.argsort()[-4:][::-1]
    print(j, termeni[indici].tolist())


## 8. Alegerea lui K cu o evaluare controlată

Silhouette măsoară separarea geometrică; nu garantează semnificația tematică. Verificăm și conținutul fiecărui grup.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

texte = ["sport tenis", "sport fotbal", "tenis meci", "python cod", "python algoritm", "date python", "reteta tort", "reteta supa", "tort prajitura"]
X = TfidfVectorizer().fit_transform(texte)
for k in (2, 3, 4):
    etichete = KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X)
    print(k, "silhouette =", round(silhouette_score(X, etichete), 3))


## 9. Reducerea dimensionalității cu TruncatedSVD (LSA)

TruncatedSVD nu centrează matricea sparse și este potrivit pentru TF-IDF. După proiecție, normalizarea L2 facilitează compararea vectorilor.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

texte = ["graf bfs muchii", "graf dfs noduri", "dinamica stare", "dp memoizare", "sir palindrom", "sir prefix"]
X = TfidfVectorizer().fit_transform(texte)
lsa = make_pipeline(TruncatedSVD(n_components=2, random_state=0), Normalizer())
X_redus = lsa.fit_transform(X)
print("Original:", X.shape, "Redus:", X_redus.shape)


## 10. MiniBatchKMeans pentru corpusuri mai mari

Folosește mini-loturi când ai multe documente și vrei timp de antrenare mai mic; verifică stabilitatea soluției.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.feature_extraction.text import TfidfVectorizer

texte = ["sport tenis", "sport fotbal", "python cod", "python algoritm", "reteta supa", "reteta tort"] * 8
X = TfidfVectorizer().fit_transform(texte)
m = MiniBatchKMeans(n_clusters=3, batch_size=16, n_init=10, random_state=3)
print(m.fit_predict(X)[:12])


## 11. NMF pe matricea TF-IDF

Matricea W conține ponderea fiecărei teme în fiecare document; H descrie temele prin termeni ponderați. Ponderile nu sunt probabilități calibrate.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

texte = ["tenis meci sport", "fotbal campionat sport", "python cod algoritm", "date python cod", "supa reteta ingrediente", "tort reteta ciocolata"]
v = TfidfVectorizer()
X = v.fit_transform(texte)
nmf = NMF(n_components=3, init="nndsvda", random_state=0, max_iter=800)
W = nmf.fit_transform(X)
H = nmf.components_
termeni = v.get_feature_names_out()
for j, topic in enumerate(H):
    print("Tema", j, termeni[topic.argsort()[-4:][::-1]].tolist())
print("Documentul 0 - ponderi:", W[0].round(3))


## 12. LDA probabilist cu numărări de termeni

LDA este conceput pentru numărări nenegative; vectorii per-document oferă proporții ale temelor ce însumează aproximativ 1.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

texte = ["tenis meci sport", "fotbal campionat sport", "python cod algoritm", "date python cod", "supa reteta ingrediente", "tort reteta ciocolata"]
v = CountVectorizer()
X = v.fit_transform(texte)
lda = LatentDirichletAllocation(n_components=3, learning_method="batch", max_iter=20, random_state=0)
proportii = lda.fit_transform(X)
termeni = v.get_feature_names_out()
for j, topic in enumerate(lda.components_):
    print("Tema", j, termeni[topic.argsort()[-4:][::-1]].tolist())
print("Proporții doc0:", proportii[0].round(3), "sumă:", round(sum(proportii[0]), 3))


## 13. Comparăm două valori ale numărului de teme

Eroarea de reconstrucție scade adesea cu mai multe teme, dar numărul potrivit depinde și de interpretabilitate și stabilitate.

In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

texte = ["graf bfs", "graf dfs", "dinamica stare", "dp memoizare", "sir prefix", "sir sufix"]
X = TfidfVectorizer().fit_transform(texte)
for k in (2, 3):
    model = NMF(n_components=k, init="nndsvda", random_state=0, max_iter=800)
    model.fit(X)
    print(k, "eroarea reconstrucției:", round(model.reconstruction_err_, 3))


## 14. Detectăm texte atipice după proiecție LSA

Scorurile sunt relative la distribuția corpusului și reprezentarea aleasă. Un text marcat ca atipic nu este automat greșit sau fraudulos.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.ensemble import IsolationForest

texte = ["graf bfs noduri", "graf dfs muchii", "graf drum minim", "dfs componente graf", "bfs drum graf", "reteta prajitura ciocolata"]
X = TfidfVectorizer().fit_transform(texte)
Y = Normalizer().fit_transform(TruncatedSVD(n_components=2, random_state=0).fit_transform(X))
m = IsolationForest(contamination=0.17, random_state=0, n_estimators=200)
scor = -m.fit(Y).score_samples(Y)
for t, s in sorted(zip(texte, scor), key=lambda x: -x[1]):
    print(f"{s:.3f}", t)


## 15. LocalOutlierFactor pentru densitate locală

LOF compară densitatea vecinilor; exemplul numeric este geometric, nu text brut. În NLP avem nevoie mai întâi de reprezentarea numerică.

In [ ]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor

X = np.array([[0, 0], [0.1, 0.2], [0.2, 0.1], [0.3, 0.2], [4, 4]], dtype=float)
lof = LocalOutlierFactor(n_neighbors=2)
print("Etichete (-1=outlier):", lof.fit_predict(X))
print("Scor negativ:", lof.negative_outlier_factor_.round(2))


## 16. Cum asociem un cluster cu distribuția unor etichete cunoscute

Etichetele cunoscute sunt folosite numai pentru analiză, nu pentru a transforma un algoritm nesupervizat în clasificator.

In [ ]:
from collections import Counter, defaultdict

etichete_adevarate = ["graphs", "graphs", "dp", "dp", "strings", "strings"]
clustere = [2, 2, 0, 0, 1, 1]
grupari = defaultdict(list)
for c, y in zip(clustere, etichete_adevarate):
    grupari[c].append(y)
for c, valori in sorted(grupari.items()):
    print("Cluster", c, Counter(valori))


## 17. Studiu ONIA #266: câmpuri text combinate într-un singur document

Datele sunt ilustrative, nu fișierul oficial. Nu introducem eticheta țintă în documentul vectorizat.

In [ ]:
import pandas as pd

# Înlocuiește cu calea fișierului oferit de platformă.
# Structura corespunde enunțului problemei #266.
exemplu = pd.DataFrame({
    "id": ["P1", "P2"],
    "title": ["Shortest path", "Prime factors"],
    "description": ["Find a path in a graph", "Factorize integers"],
    "statement": ["Given vertices and edges...", "Given an integer n..."]
})
exemplu["text"] = (
    exemplu["title"].fillna("") + " " +
    exemplu["description"].fillna("") + " " +
    exemplu["statement"].fillna("")
)
print(exemplu[["id", "text"]].to_string(index=False))


## 18. ONIA #266: NMF ca sursă de caracteristici auxiliare

Este o demonstrație pe trei clase sintetice, nu un scor pentru problema oficială cu cinci clase. NMF extrage trăsături; LogisticRegression realizează clasificarea.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

train_text = ["graph bfs edges", "graph dfs nodes", "dynamic programming memoization", "dp states transitions", "string prefix palindrome", "string suffix pattern"]
y_train = ["graphs", "graphs", "dp", "dp", "strings", "strings"]
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    NMF(n_components=3, init="nndsvda", max_iter=600, random_state=0),
    LogisticRegression(max_iter=1000)
)
model.fit(train_text, y_train)
print(model.predict(["graph shortest path edges", "string prefix search"]))


## 19. ONIA #266: validare Macro-F1 pentru un baseline supervizat

Un baseline supervizat este punctul corect de comparație. Pentru #266 folosește cele cinci clase și un set de validare fără leakuri.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

texte = ["graph bfs", "graph dfs", "graph shortest path", "graph edges", "dp memoization", "dp table", "dynamic states", "dynamic transitions", "string pattern", "string prefix", "string substring", "string suffix"]
y = ["graphs"] * 4 + ["dp"] * 4 + ["strings"] * 4
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=0)
print(cross_val_score(model, texte, y, cv=cv, scoring="f1_macro"))


## 20. Numărul de parametri și dimensiunea spațiului de caracteristici

N-gramele pot ajuta semantic, dar cresc rapid vocabularul și costul memoriei. Matricile rare stochează doar valorile nenule.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texte = ["graph graph edges", "graph path", "dynamic states", "string prefix"]
for config in [(1, 1), (1, 2)]:
    model = TfidfVectorizer(ngram_range=config, min_df=1)
    X = model.fit_transform(texte)
    print("ngram_range=", config, "shape=", X.shape, "nnz=", X.nnz)
